# 诗歌生成

# 数据处理

In [26]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset(file_path='poems.txt'):
    instances, word2id, id2word = process_dataset(file_path)
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [27]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    @tf.function
    def call(self, inp_ids):
        x = self.embed_layer(inp_ids)          # (batch, seq_len, 64)
        rnn_out = self.rnn_layer(x)            # (batch, seq_len, 128)
        logits = self.dense(rnn_out)           # (batch, seq_len, v_sz)
        return logits
    
    @tf.function
    def get_next_token(self, x, state):
        inp_emb = self.embed_layer(x)          # (batch, 64)
        h, state = self.rnncell(inp_emb, state) # (batch, 128), (batch, 128)
        logits = self.dense(h)                 # (batch, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [28]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [29]:
@tf.function
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

@tf.function
def train_one_step(model, optimizer, x, y, seqlen):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y, seqlen)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [30]:
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()   # 读取当前目录下的 poems.txt
model = myRNNModel(word2id)

# 关键：显式构建模型，避免梯度为空
model.build(input_shape=(None, None))   # 输入形状为 (batch, seq_len)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

e:\Anaconda3\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'my_rnn_model_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


epoch 0 : loss 8.821366
epoch 1 : loss 6.551993
epoch 2 : loss 6.0268307
epoch 3 : loss 5.8291755
epoch 4 : loss 5.5590734
epoch 5 : loss 5.544198
epoch 6 : loss 5.412344
epoch 7 : loss 5.3639765
epoch 8 : loss 5.2574673
epoch 9 : loss 5.2123322


# 生成过程

In [31]:
def gen_poem(start_word='日', max_len=50):
    # 初始化 state 为零向量
    state = tf.zeros((1, 128), dtype=tf.float32)
    
    # 处理起始词
    if start_word not in word2id:
        start_word = 'UNK'
    cur_token = tf.constant([word2id[start_word]], dtype=tf.int32)
    
    collect = [start_word]
    for _ in range(max_len - 1):
        cur_token, state = model.get_next_token(cur_token, state)
        token_id = cur_token.numpy()[0]
        word = id2word[token_id]
        if word == end_token:
            break
        collect.append(word)
        cur_token = tf.constant([token_id], dtype=tf.int32)
    
    return ''.join(collect)

# 测试
for start in ['日', '红', '山', '夜', '湖', '海', '月']:
    print(f"{start}: {gen_poem(start)}")

日: 日暮风吹落，花落花花，一枝红叶落花开。
红: 红蕉柳絮啼花落夜寒花。
山: 山畔夜来来处，一枝春雨满庭花。
夜: 夜暮风吹，一枝春月。
湖: 湖畔夜，秋风落日深。
海: 海滨，一枝一夜深。
月: 月落日春，一枝花下水。
